# PySEAL Implementation Example - One

In [1]:
import seal

from seal import Ciphertext
from seal import Decryptor
from seal import Encryptor
from seal import EncryptionParameters
from seal import Evaluator
from seal import IntegerEncoder
from seal import KeyGenerator
from seal import Plaintext
from seal import SEALContext

## Setting Encryption Parameters

### poly_modulus

- Polynomial modulus size, defines the polynomial ring for computations. Larger degrees -> more noise buget but slower computation

### coeff_modulus

- Coefficient modulus determines the ciphertext space

### plain_modulus

- Plaintext modulus is for encoding messages. 1 << 8 = 256 means messages are modulo 256

In [2]:
parms = EncryptionParameters()

parms.set_poly_modulus("1x^2048 + 1")

parms.set_coeff_modulus(seal.coeff_modulus_128(2048))

parms.set_plain_modulus(1 << 8)


## Create Context and Show Parameters

Checks and stores all parameters, makes print to visualize

In [3]:
context = SEALContext(parms)

print("Encryption parameters:")
print("! poly_modulus: " + context.poly_modulus().to_string())
print("! coeff_modulus_size: " + (str)(context.total_coeff_modulus().significant_bit_count()) + " bits")
print("! plain_modulus: " + (str)(context.plain_modulus().value()))
print("! noise_standard_deviation: " + (str)(context.noise_standard_deviation()))

Encryption parameters:
! poly_modulus: 1x^2048 + 1
! coeff_modulus_size: 56 bits
! plain_modulus: 256
! noise_standard_deviation: 3.19


## Integer Encoding and Key Generation/Encryption Tools

Converts integers into plaintext polynomials compatible with the SEAL backend. SEAL works over polynomials, so raw integers must be encoded first.

Also generating a public/secret key pair and encryptor, evaluator, and decryptor objects.

In [4]:
encoder = IntegerEncoder(context.plain_modulus())

keygen = KeyGenerator(context)
public_key = keygen.public_key()
secret_key = keygen.secret_key()

encryptor = Encryptor(context, public_key)
evaluator = Evaluator(context)
decryptor = Decryptor(context, secret_key)

In [5]:
value1 = 2
plain1 = encoder.encode(value1);
print("Encoded " + (str)(value1) + " as polynomial " + plain1.to_string() + " (plain1)")

Encoded 2 as polynomial 1x^1 (plain1)


In [6]:
value2 = 4
plain2 = encoder.encode(value2);
print("Encoded " + (str)(value2) + " as polynomial " + plain2.to_string() + " (plain2)")

Encoded 4 as polynomial 1x^2 (plain2)


## Encrypt the two plaintexts

Both integers are now turned into encrypted ciphertexts

In [7]:
encrypted1 = Ciphertext()
encrypted2 = Ciphertext()
print("Encrypting plain1: ")
encryptor.encrypt(plain1, encrypted1)
print("Done (encrypted1)")
print("THIS IS ENCRYPTED 1: ", encrypted1)

print("Encrypting plain2: ")
encryptor.encrypt(plain2, encrypted2)
print("Done (encrypted2)")
print("THIS IS ENCRYPTED 2: ", encrypted2)

Encrypting plain1: 
Done (encrypted1)
THIS IS ENCRYPTED 1:  <seal.Ciphertext object at 0x107932430>
Encrypting plain2: 
Done (encrypted2)
THIS IS ENCRYPTED 2:  <seal.Ciphertext object at 0x1079312f0>


## Observing the Noise Budger

The noise budget shows how many operations remain before decryption fails. Operations increase noise, which it gets too large, the ciphertext becomes undecryptable

In [8]:
print("Noise budget in encrypted1: " + (str)(decryptor.invariant_noise_budget(encrypted1)) + " bits")
print("Noise budget in encrypted2: " + (str)(decryptor.invariant_noise_budget(encrypted2)) + " bits")

Noise budget in encrypted1: 37 bits
Noise budget in encrypted2: 37 bits


## Homomorphic Operations

Performing negation, addition and multiplication. All of these are done while the values are encrypted. Noise buget is checked after each operation to monitor degradation

In [9]:
evaluator.negate(encrypted1)
print("Noise budget in -encrypted1: " + (str)(decryptor.invariant_noise_budget(encrypted1)) + " bits")

Noise budget in -encrypted1: 37 bits


In [10]:
evaluator.add(encrypted1, encrypted2)
print("Noise budget in -encrypted1 + encrypted2: " + (str)(decryptor.invariant_noise_budget(encrypted1)) + " bits")

Noise budget in -encrypted1 + encrypted2: 37 bits


In [11]:
evaluator.multiply(encrypted1, encrypted2)
print("Noise budget in (-encrypted1 + encrypted2) * encrypted2: " + (str)(decryptor.invariant_noise_budget(encrypted1)) + " bits")

Noise budget in (-encrypted1 + encrypted2) * encrypted2: 19 bits


In [12]:
plain_result = Plaintext()
print("Decrypting result: ")
decryptor.decrypt(encrypted1, plain_result)
print("Done")
print("Plaintext polynomial: " + plain_result.to_string())

Decrypting result: 
Done
Plaintext polynomial: 1x^4 + FFx^3


In [13]:
print("Decoded integer: " + (str)(encoder.decode_int32(plain_result)))

Decoded integer: 8
